In [5]:
from geopy.geocoders import Nominatim

# Initialize the geolocator
geolocator = Nominatim(user_agent="geoapi")

# Provide latitude and longitude
latitude = 19.3810601
longitude = -99.1880785

# Perform reverse geocoding
location = geolocator.reverse((latitude, longitude))

# Display the address
if location:
    print("Address:", location.address)
    print("Address:", location)
    # Get the colony or neighborhood from the address if available
    address_details = location.raw.get("address", {})
    colony = address_details.get("neighbourhood")  # May vary depending on the address structure
    print("Colony (Neighborhood):", colony)
    print("address_details (Neighborhood):", address_details)
else:
    print("No location found!")

Address: Los Caporales, 85, Calle Miguel Ángel Buonarroti, Colonia Nonoalco, Ciudad de México, Benito Juárez, Ciudad de México, 03700, México
Address: Los Caporales, 85, Calle Miguel Ángel Buonarroti, Colonia Nonoalco, Ciudad de México, Benito Juárez, Ciudad de México, 03700, México
Colony (Neighborhood): Colonia Nonoalco
address_details (Neighborhood): {'amenity': 'Los Caporales', 'house_number': '85', 'road': 'Calle Miguel Ángel Buonarroti', 'neighbourhood': 'Colonia Nonoalco', 'city': 'Ciudad de México', 'borough': 'Benito Juárez', 'state': 'Ciudad de México', 'ISO3166-2-lvl4': 'MX-CMX', 'postcode': '03700', 'country': 'México', 'country_code': 'mx'}


In [17]:
from geopy.distance import geodesic

# Define your starting latitude and longitude
latitude = 19.3810601
longitude = -99.1880785

# Distance to add (in meters)
distance_in_meters = 30

# Function to calculate new coordinates
def add_distance(lat, lon, distance, bearing):
    from math import radians, degrees, sin, cos, atan2
    
    # Earth radius (in meters)
    R = 6378137
    
    # Convert latitude and longitude from degrees to radians
    lat_rad = radians(lat)
    lon_rad = radians(lon)
    bearing_rad = radians(bearing)
    
    # Calculate new latitude and longitude
    new_lat = lat_rad + (distance / R) * cos(bearing_rad)
    new_lon = lon_rad + (distance / R) * sin(bearing_rad) / cos(lat_rad)
    
    # Convert back to degrees
    new_lat = degrees(new_lat)
    new_lon = degrees(new_lon)
    
    return new_lat, new_lon

# Add 5 meters in different directions (e.g., bearings 0°, 90°, 180°, 270°)
directions = ["North", "East", "South", "West"]
bearings = [0, 90, 180, 270]

polygon_coords = []
for direction, bearing in zip(directions, bearings):
    new_lat, new_lon = add_distance(latitude, longitude, distance_in_meters, bearing)
    lat_long = [new_lat,new_lon]
    print(lat_long)
    polygon_coords.append(lat_long)
print(polygon_coords)

[19.381329594585235, -99.18807850000002]
[19.3810601, -99.1877928164135]
[19.380790605414763, -99.18807850000002]
[19.3810601, -99.18836418358651]
[[19.381329594585235, -99.18807850000002], [19.3810601, -99.1877928164135], [19.380790605414763, -99.18807850000002], [19.3810601, -99.18836418358651]]


In [24]:
import folium

# Create a map centered on Mexico City
map_mx = folium.Map(location=[latitude, longitude], zoom_start=17)

# Define the polygon coordinates


# Add a semi-transparent polygon to create a shadow effect
folium.Polygon(
    locations=polygon_coords,
    color="black",
    fill=True,
    fill_color="gray",
    fill_opacity=0.3  # Shadow transparency
).add_to(map_mx)

# Display the map inside Jupyter Notebook
map_mx

In [32]:
import gcsfs
import pandas as pd
import bigframes.pandas as bpd
from google.cloud import storage
from google.oauth2 import service_account
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

PROJECT_ID = 'acme-987654'
LOCATION = "southamerica-east1" 
service_account_path = 'acme-987654-c052039ac4cd.json'

bucket_name = "acme_storage"

dataset_curated = "Curated"

table_checkin_id = "CheckIn"
table_user_id = "User" 
table_business_id = "Business"
table_review_id ="Review"
table_sitios_id = "Sitios"

table_estados_california_id = "Sitios_California"
table_estados_carolina_id = "Sitios_Carolina_Sur"
table_estados_florida_id = "Sitios_Florida"
table_estados_Massachusetts_id = "Sitios_Massachusetts"
table_estados_Texas_id = "Sitios_Texas"
table_estados_Washington_id = "Sitios_Washington"
table_estados_Luisiana_id = "Sitios_Luisiana"


In [39]:
def get_table_curated_by_columns(table, columns,city):
    # Initialize BigQuery client
    client = bigquery.Client.from_service_account_json(service_account_path)

    # Define your query
    query = f"""
        SELECT {columns} FROM acme-987654.Curated.{table} where city = {city}
    """

    
    # Run the query
    print(query)
    query_job = client.query(query)
    df = query_job.to_dataframe()
    return df

In [40]:
df_lat_long = get_table_curated_by_columns("Estados", "latitude, longitude","'new orleans'")


        SELECT latitude, longitude FROM acme-987654.Curated.Estados where city = 'new orleans'
    


In [41]:
df_lat_long

,latitude,longitude
0,32.155983,-80.757296
1,32.155983,-80.757296
2,32.155983,-80.757296
3,32.155983,-80.757296
4,32.155983,-80.757296
...,...,...
15521,29.952991,-90.070463
15522,29.952991,-90.070463
15523,29.952991,-90.070463
15524,29.952991,-90.070463


In [42]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.cluster import KMeans

# Combine latitude and longitude into a 2D array
coordinates = np.vstack([df_lat_long.latitude, df_lat_long.longitude])

# Perform Kernel Density Estimation (KDE)
kde = gaussian_kde(coordinates)
density = kde(coordinates)  # Calculate density for each point

# Use KMeans to cluster the high-density points
num_clusters = 1  # You can change this based on your needs
kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(coordinates.T)  # .T transposes the array
cluster_centers = kmeans.cluster_centers_

# Recommended latitude and longitude (centroid of the cluster)
recommended_latitude = cluster_centers[0, 0]
recommended_longitude = cluster_centers[0, 1]
print(f"Recommended Location: Latitude = {recommended_latitude}, Longitude = {recommended_longitude}")



Recommended Location: Latitude = 30.022618734690194, Longitude = -90.19571534296018
